In [ ]:
# ============================================================================
#  Random Forest activity classifier - 5-fold subject-wise cross-validation
# ----------------------------------------------------------------------------
#  A forest cannot consume a (128, 6) time series, so every segment is collapsed
#  into one ~213-dimensional feature vector. Feature engineering is therefore the
#  whole model here: these features are the ceiling.
#
#  Data sources, deliberately mixed:
#    TRAIN  segmented_4s/         - natural distribution, NO augmentation.
#                                   A forest handles skew with class_weight, so
#                                   the synthetic 16% that the CNN needed is a
#                                   liability here: augmented segments are near
#                                   duplicates and trees will split on them.
#    VAL    balanced_folds/*/X_val   } already non-overlapping and at the natural
#    TEST   balanced_folds/*/X_test  } distribution - reused verbatim so this
#                                      model is scored on byte-identical data to
#                                      the CNN and the comparison is exact.
#
#  Splits come from updated_cv_5_folds/ - the same subject-wise fold lists.
#  No normalisation: trees split on thresholds and are scale-invariant.
# ============================================================================
import json
import os
import time

import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp
from sklearn.ensemble import RandomForestClassifier

SEG_DIR   = "segmented_4s"
FOLD_DIR  = "updated_cv_5_folds"
BAL_DIR   = "balanced_folds"
CACHE_DIR = "rf_features"
OUT_DIR   = "rf_results"

N_FOLDS   = 5
N_CLASSES = 7
SEG_LEN   = 128
FS        = 32.0                     # Hz
CHANNELS  = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]

TRAIN_CAP_PER_CLASS = 120_000        # undersample the majority; no augmentation
CHUNK               = 40_000         # segments per feature-extraction batch
SEED                = 0

RF_PARAMS = dict(n_estimators=200, min_samples_leaf=4, max_features="sqrt",
                 class_weight="balanced", n_jobs=24, random_state=SEED)

CLASS_NAMES = ["Lying down", "Sitting", "Walking", "Running",
               "Bicycling", "Standing in place", "Standing and moving"]

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print(f"sklearn RandomForest | {RF_PARAMS['n_estimators']} trees, "
      f"min_samples_leaf={RF_PARAMS['min_samples_leaf']}, "
      f"class_weight={RF_PARAMS['class_weight']}")


def fold_users(i, part):
    with open(f"{FOLD_DIR}/fold_{i}_{part}_uuids.txt") as fh:
        return [l.strip() for l in fh if l.strip()]

In [ ]:
# ---------------------------------------------------------------------------
#  Feature extraction: (N, 128, 6) -> (N, 213)
# ---------------------------------------------------------------------------
#  The 6 raw channels are first expanded to 10 signals. The four derived ones
#  are the point of the exercise:
#     |acc|, |gyro|        rotation-invariant - unchanged by how the phone sits
#     acc_vertical         acceleration along the estimated gravity direction
#     acc_horizontal       the component perpendicular to it
#  Gravity is estimated as the segment mean, which at 4 s is dominated by the
#  static component. Splitting motion into vertical and horizontal converts a
#  device-relative description into a world-relative one: walking oscillates
#  vertically, cycling mostly horizontally, and that holds across users in a way
#  raw acc_z does not.
# ---------------------------------------------------------------------------

SIGNAL_NAMES = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z",
                "acc_mag", "gyro_mag", "acc_vert", "acc_horiz"]
FREQS = np.fft.rfftfreq(SEG_LEN, d=1.0 / FS)          # 65 bins, 0 .. 16 Hz
BANDS = [(0.0, 0.5), (0.5, 3.0), (3.0, 8.0), (8.0, 16.0)]
AC_LO, AC_HI = 8, 64                                   # lags: 4 Hz down to 0.5 Hz


def _signals(X):
    """(N,128,6) -> (N,128,10): raw channels plus the four derived signals."""
    acc, gyr = X[:, :, :3], X[:, :, 3:]
    am = np.linalg.norm(acc, axis=2)
    gm = np.linalg.norm(gyr, axis=2)
    g = acc.mean(axis=1, keepdims=True)                       # gravity estimate
    gn = g / np.maximum(np.linalg.norm(g, axis=2, keepdims=True), 1e-8)
    vert = (acc * gn).sum(axis=2)                             # along gravity
    horiz = np.linalg.norm(acc - vert[:, :, None] * gn, axis=2)
    return np.concatenate([X, am[:, :, None], gm[:, :, None],
                           vert[:, :, None], horiz[:, :, None]], axis=2), gn[:, 0, :]


def _feature_names():
    n = []
    for stat in ["mean", "std", "min", "max", "median", "p25", "p75", "rms", "mad",
                 "skew", "kurt", "zcr", "jerk_mean", "jerk_std",
                 "dom_freq", "dom_power", "spec_energy", "spec_entropy", "spec_centroid"]:
        n += [f"{s}_{stat}" for s in SIGNAL_NAMES]
    for s in ["acc_mag", "gyro_mag"]:
        n += [f"{s}_band{lo}-{hi}Hz" for lo, hi in BANDS]
        n += [f"{s}_ac_peak", f"{s}_ac_lag"]
    n += ["corr_acc_xy", "corr_acc_xz", "corr_acc_yz",
          "corr_gyr_xy", "corr_gyr_xz", "corr_gyr_yz", "corr_accmag_gyrmag"]
    n += ["grav_mag", "grav_dir_x", "grav_dir_y", "grav_dir_z"]
    return n


FEATURE_NAMES = _feature_names()


def extract_features(X):
    """X: (N, 128, 6) float32 -> (N, len(FEATURE_NAMES)) float32."""
    S, gdir = _signals(np.asarray(X, dtype=np.float32))        # (N,128,10)
    mu = S.mean(1)
    sd = S.std(1)
    dev = S - mu[:, None, :]

    # --- A: distribution, B: shape, C: dynamics -----------------------------
    diff = np.abs(np.diff(S, axis=1))
    sd_safe = np.maximum(sd, 1e-8)
    parts = [mu, sd, S.min(1), S.max(1), np.median(S, 1),
             np.percentile(S, 25, axis=1), np.percentile(S, 75, axis=1),
             np.sqrt((S ** 2).mean(1)), np.abs(dev).mean(1),
             (dev ** 3).mean(1) / sd_safe ** 3,                  # skewness
             (dev ** 4).mean(1) / sd_safe ** 4 - 3.0,            # excess kurtosis
             (np.diff(np.signbit(dev), axis=1) != 0).mean(1),    # zero crossings
             diff.mean(1), diff.std(1)]

    # --- D: spectral --------------------------------------------------------
    F = np.abs(np.fft.rfft(dev, axis=1))                        # (N,65,10)
    P = F ** 2
    P1 = P[:, 1:, :]                                            # drop DC
    k = P1.argmax(1)                                            # (N,10)
    tot = np.maximum(P1.sum(1), 1e-12)
    Pn = P1 / tot[:, None, :]
    parts += [FREQS[1:][k],                                     # dominant freq
              np.take_along_axis(P1, k[:, None, :], 1)[:, 0, :],  # its power
              tot,
              -(Pn * np.log(Pn + 1e-12)).sum(1),                # spectral entropy
              (FREQS[1:, None] * P1).sum(1) / tot]              # spectral centroid

    # --- E: band energy, F: periodicity (magnitudes only) -------------------
    for idx in (6, 7):                                          # acc_mag, gyro_mag
        p = P[:, :, idx]
        parts += [np.stack([p[:, (FREQS >= lo) & (FREQS < hi)].sum(1)
                            for lo, hi in BANDS], axis=1)]
        ac = np.fft.irfft(p, n=SEG_LEN, axis=1)                 # autocorrelation
        ac = ac / np.maximum(ac[:, :1], 1e-12)
        seg = ac[:, AC_LO:AC_HI]
        lag = seg.argmax(1)
        parts += [np.stack([seg.max(1), (lag + AC_LO).astype(np.float32)], axis=1)]

    # --- G: cross-axis correlation -----------------------------------------
    def corr(a, b):
        za, zb = S[:, :, a] - mu[:, None, a], S[:, :, b] - mu[:, None, b]
        return (za * zb).mean(1) / np.maximum(sd[:, a] * sd[:, b], 1e-8)
    parts += [np.stack([corr(0, 1), corr(0, 2), corr(1, 2),
                        corr(3, 4), corr(3, 5), corr(4, 5), corr(6, 7)], axis=1)]

    # --- H: orientation -----------------------------------------------------
    parts += [np.linalg.norm(mu[:, :3], axis=1, keepdims=True), gdir]

    out = np.concatenate([p if p.ndim == 2 else p[:, None] for p in parts], axis=1)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def extract_chunked(X, chunk=CHUNK):
    """Extract in batches - the (N,128,10) intermediate is 5 KB per segment."""
    return np.concatenate([extract_features(X[s:s + chunk])
                           for s in range(0, len(X), chunk)], axis=0)


print(f"{len(FEATURE_NAMES)} features from {len(SIGNAL_NAMES)} signals")
_t = extract_features(np.random.randn(4, SEG_LEN, 6).astype("float32"))
print(f"  shape check: {_t.shape}   all finite: {bool(np.isfinite(_t).all())}")

In [ ]:
# ---------------------------------------------------------------------------
#  Assembling a fold
# ---------------------------------------------------------------------------
#  TRAIN comes from segmented_4s at its natural distribution (125:1), capped per
#  class. The cap is allocated EQUALLY ACROSS USERS with redistribution, not by
#  random sampling: a random draw down to 120k Sitting segments would preserve
#  the per-user skew exactly, whereas equal quotas also flatten the heavy
#  contributors. Rare classes fall under the cap and are kept whole.
#
#  VAL and TEST are read straight from balanced_folds, where they were already
#  written as non-overlapping segments at the natural distribution.
# ---------------------------------------------------------------------------

def water_fill(avail, target):
    """Equal per-user quota, redistributing what short users cannot supply."""
    avail = np.asarray(avail, dtype=np.int64)
    take = np.zeros_like(avail)
    remaining, active = int(min(target, avail.sum())), avail > 0
    while remaining > 0 and active.any():
        share = max(1, remaining // int(active.sum()))
        give = np.minimum(share, avail - take)
        give[~active] = 0
        if give.sum() == 0:
            break
        if give.sum() > remaining:
            for j in np.flatnonzero(give):
                give[j] = min(give[j], max(remaining, 0))
                remaining -= give[j]
            take += give
            break
        take += give
        remaining -= int(give.sum())
        active = (avail - take) > 0
    return take


def _load_user_segments(u):
    d = pd.read_parquet(f"{SEG_DIR}/{u}.parquet",
                        columns=["window", "Activity_Labels"] + CHANNELS)
    n = len(d) // SEG_LEN
    X = d[CHANNELS].to_numpy(dtype=np.float32).reshape(n, SEG_LEN, len(CHANNELS))
    y = d["Activity_Labels"].to_numpy()[::SEG_LEN] - 1          # 1-7 -> 0-6
    w = d["window"].to_numpy()[::SEG_LEN]
    return X, y, w


def _count_user(u):
    d = pd.read_parquet(f"{SEG_DIR}/{u}.parquet", columns=["Activity_Labels"])
    y = d["Activity_Labels"].to_numpy()[::SEG_LEN] - 1
    return u, np.bincount(y, minlength=N_CLASSES)


def build_train(i, rng):
    """Feature matrix for one fold's training users, capped per class."""
    users = fold_users(i, "train")
    ctx = mp.get_context("fork")
    with ProcessPoolExecutor(max_workers=12, mp_context=ctx) as pool:
        counts = dict(pool.map(_count_user, users))
    avail = np.stack([counts[u] for u in users])                # (n_users, 7)
    quota = np.stack([water_fill(avail[:, k], TRAIN_CAP_PER_CLASS)
                      for k in range(N_CLASSES)], axis=1)       # (n_users, 7)

    Fs, ys, us = [], [], []
    for j, u in enumerate(users):
        X, y, _ = _load_user_segments(u)
        pick = []
        for k in range(N_CLASSES):
            q = int(quota[j, k])
            if q <= 0:
                continue
            idx = np.flatnonzero(y == k)
            if len(idx) > q:                       # spread across the session
                idx = idx[np.linspace(0, len(idx) - 1, q).round().astype(int)]
            pick.append(idx)
        if not pick:
            continue
        pick = np.sort(np.concatenate(pick))
        Fs.append(extract_chunked(X[pick]))
        ys.append(y[pick])
        us.append(np.full(len(pick), j, dtype=np.int16))
        del X, y
    return np.concatenate(Fs), np.concatenate(ys), np.concatenate(us)


def build_eval(i, part):
    """Features for val/test, reused verbatim from balanced_folds."""
    X = np.load(f"{BAL_DIR}/fold_{i}/X_{part}.npy", mmap_mode="r")
    y = np.load(f"{BAL_DIR}/fold_{i}/y_{part}.npy")
    u = np.load(f"{BAL_DIR}/fold_{i}/u_{part}.npy")
    w = np.load(f"{BAL_DIR}/fold_{i}/w_{part}.npy")
    F = np.concatenate([extract_features(np.asarray(X[s:s + CHUNK]))
                        for s in range(0, len(X), CHUNK)], axis=0)
    return F, y, u, w


def cached_fold(i):
    """Extract once, reuse for every later experiment."""
    path = f"{CACHE_DIR}/fold_{i}.npz"
    if os.path.exists(path):
        z = np.load(path)
        return {k: z[k] for k in z.files}
    rng = np.random.default_rng(SEED + i)
    t0 = time.time()
    Xtr, ytr, utr = build_train(i, rng)
    Xva, yva, uva, wva = build_eval(i, "val")
    Xte, yte, ute, wte = build_eval(i, "test")
    d = dict(Xtr=Xtr, ytr=ytr, utr=utr, Xva=Xva, yva=yva, uva=uva, wva=wva,
             Xte=Xte, yte=yte, ute=ute, wte=wte)
    np.savez(path, **d)
    print(f"  fold {i}: train {Xtr.shape}  val {Xva.shape}  test {Xte.shape}"
          f"   ({time.time() - t0:.0f}s, cached)")
    return d

In [ ]:
# ---------------------------------------------------------------------------
#  Metrics, computed from the confusion matrix
# ---------------------------------------------------------------------------

def confusion_np(y_true, y_pred, n=N_CLASSES):
    cm = np.zeros((n, n), dtype=np.int64)
    np.add.at(cm, (np.asarray(y_true), np.asarray(y_pred)), 1)
    return cm                                    # rows = true, cols = predicted


def class_metrics(cm):
    tp = np.diag(cm).astype(np.float64)
    support, predicted, total = cm.sum(1), cm.sum(0), cm.sum()
    recall = np.where(support > 0, tp / np.maximum(support, 1), np.nan)
    precision = np.where(predicted > 0, tp / np.maximum(predicted, 1), 0.0)
    r0 = np.nan_to_num(recall)
    f1 = np.where(precision + r0 > 0,
                  2 * precision * r0 / np.maximum(precision + r0, 1e-12), 0.0)
    f1 = np.where(support > 0, f1, np.nan)
    specificity = (total - support - (predicted - tp)) / np.maximum(total - support, 1)
    return {"recall": recall, "precision": precision, "f1": f1,
            "bal_acc": (r0 + specificity) / 2,           # one-vs-rest
            "ovr_acc": (total - (support - tp) - (predicted - tp)) / total,
            "support": support}


def overall_metrics(cm):
    c = class_metrics(cm)
    total = cm.sum()
    accuracy = np.diag(cm).sum() / total
    p_e = float((cm.sum(1) * cm.sum(0)).sum()) / (total * total)
    return {"accuracy": float(accuracy),
            "macro_f1": float(np.nanmean(c["f1"])),
            "balanced_accuracy": float(np.nanmean(c["recall"])),
            "kappa": float((accuracy - p_e) / (1 - p_e)) if p_e < 1 else 0.0}

In [ ]:
# ---------------------------------------------------------------------------
#  Train and evaluate every fold
# ---------------------------------------------------------------------------
#  Validation is scored but not tuned against here - a forest has no early
#  stopping, so val exists to compare hyper-parameter settings between runs.
#  Change RF_PARAMS, watch validation, and only then read the test numbers.
# ---------------------------------------------------------------------------

def run_fold(i, verbose=True):
    d = cached_fold(i)
    t0 = time.time()
    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(d["Xtr"], d["ytr"])
    fit_s = time.time() - t0

    out = {"fold": i, "fit_seconds": fit_s,
           "n_train": int(len(d["ytr"])), "n_val": int(len(d["yva"])),
           "n_test": int(len(d["yte"])),
           "importances": rf.feature_importances_.astype(np.float64)}
    for part, (X, y) in (("val", (d["Xva"], d["yva"])), ("test", (d["Xte"], d["yte"]))):
        p = rf.predict(X)
        cm = confusion_np(y, p)
        out[part] = overall_metrics(cm)
        out[f"{part}_cm"] = cm
        if part == "test":
            out["y_true"], out["y_pred"], out["w_test"] = y, p, d["wte"]

    if verbose:
        print(f"  fold {i}: fit {fit_s:.0f}s on {out['n_train']:,} segments  |  "
              f"VAL macroF1 {out['val']['macro_f1']:.4f}  ->  "
              f"TEST acc {out['test']['accuracy']:.4f}  "
              f"macroF1 {out['test']['macro_f1']:.4f}  "
              f"bal-acc {out['test']['balanced_accuracy']:.4f}  "
              f"kappa {out['test']['kappa']:.4f}")
    del d, rf
    return out


print("building features (first run extracts and caches; later runs reload)\n")
rf_results = []
t_all = time.time()
for i in range(N_FOLDS):
    rf_results.append(run_fold(i))
print(f"\nall folds complete in {(time.time() - t_all) / 60:.1f} min")

np.savez(f"{OUT_DIR}/predictions.npz",
         **{f"y_true_{r['fold']}": r["y_true"] for r in rf_results},
         **{f"y_pred_{r['fold']}": r["y_pred"] for r in rf_results},
         **{f"w_test_{r['fold']}": r["w_test"] for r in rf_results},
         **{f"importances_{r['fold']}": r["importances"] for r in rf_results})
json.dump([{k: r[k] for k in ("fold", "n_train", "n_test", "fit_seconds")}
           | {"val": r["val"], "test": r["test"]} for r in rf_results],
          open(f"{OUT_DIR}/folds.json", "w"), indent=1)
print(f"saved -> {OUT_DIR}/folds.json, {OUT_DIR}/predictions.npz")

In [ ]:
# ---------------------------------------------------------------------------
#  Overall results: per fold, then mean +/- standard deviation
# ---------------------------------------------------------------------------
METRICS = [("accuracy", "Accuracy"), ("macro_f1", "Macro F1"),
           ("balanced_accuracy", "Balanced accuracy"), ("kappa", "Cohen's kappa")]
vals = {k: np.array([r["test"][k] for r in rf_results]) for k, _ in METRICS}

print("TEST metrics per fold\n")
print(f"  {'fold':>4} {'n_train':>10} {'n_test':>10}  "
      + "".join(f"{lab:>19s}" for _, lab in METRICS))
print("  " + "-" * 102)
for r in rf_results:
    print(f"  {r['fold']:>4} {r['n_train']:>10,} {r['n_test']:>10,}  "
          + "".join(f"{r['test'][k]:19.4f}" for k, _ in METRICS))
print("  " + "-" * 102)
print(f"  {'mean':>4} {'':>10} {'':>10}  "
      + "".join(f"{vals[k].mean():19.4f}" for k, _ in METRICS))
print(f"  {'std':>4} {'':>10} {'':>10}  "
      + "".join(f"{vals[k].std(ddof=1):19.4f}" for k, _ in METRICS))

print("\n" + "=" * 74)
print("SUMMARY  (mean +/- std over 5 subject-wise folds)\n")
for k, lab in METRICS:
    print(f"  {lab:20s} {vals[k].mean():.4f}  +/-  {vals[k].std(ddof=1):.4f}"
          f"     [min {vals[k].min():.4f}, max {vals[k].max():.4f}]")
print("\n  baselines on this data:")
print("    always-predict-Sitting        accuracy 0.441   macro-F1 0.087   kappa 0.000")
print("    only the 2 majority classes   accuracy 0.787   macro-F1 0.258")
try:
    cnn = json.load(open("cnn_results/folds.json"))
    c = {k: np.array([f["test"][k] for f in cnn]) for k, _ in METRICS}
    print("\n  CNN on the same folds, for comparison:")
    for k, lab in METRICS:
        d = vals[k].mean() - c[k].mean()
        print(f"    {lab:20s} CNN {c[k].mean():.4f}   RF {vals[k].mean():.4f}   "
              f"{'RF +' if d >= 0 else 'RF '}{d:+.4f}")
except FileNotFoundError:
    pass

In [ ]:
# ---------------------------------------------------------------------------
#  TABLE 1 - per-class metrics, pooled over all five folds
# ---------------------------------------------------------------------------
#  Every user is tested exactly once across the folds, so concatenating the
#  predictions gives one estimate over all 56 users.
#
#  "accuracy", "macro F1" and "balanced accuracy" are aggregates over all
#  classes - there is no macro-F1 for a single class. The per-class equivalents
#  are recall, precision and F1. Two one-vs-rest columns are included as well:
#     bal-acc  (recall + specificity) / 2, treating the class as one-vs-rest
#     OvR acc  (TP + TN) / total for that class against all others
#  OvR accuracy flatters rare classes - it reads high for Running simply because
#  99.6% of segments are not Running. Read F1 and recall.
# ---------------------------------------------------------------------------
cm_pooled = sum(confusion_np(r["y_true"], r["y_pred"]) for r in rf_results)
c = class_metrics(cm_pooled)
o = overall_metrics(cm_pooled)

print(f"TABLE 1 - per-class metrics, pooled over {N_FOLDS} folds "
      f"({cm_pooled.sum():,} test segments)\n")
print(f"{'idx':>3}  {'activity':22s} {'recall':>8s} {'precision':>10s} {'F1':>8s} "
      f"{'bal-acc':>9s} {'OvR acc':>9s} {'support':>11s}")
print("-" * 86)
for k in range(N_CLASSES):
    print(f"{k:>3}  {CLASS_NAMES[k]:22s} {c['recall'][k]:8.3f} {c['precision'][k]:10.3f} "
          f"{c['f1'][k]:8.3f} {c['bal_acc'][k]:9.3f} {c['ovr_acc'][k]:9.3f} "
          f"{c['support'][k]:11,}")
print("-" * 86)
print(f"{'':>3}  {'macro average':22s} {np.nanmean(c['recall']):8.3f} "
      f"{c['precision'].mean():10.3f} {np.nanmean(c['f1']):8.3f} "
      f"{c['bal_acc'].mean():9.3f}")
print(f"{'':>3}  {'overall':22s} {'':>8s} {'':>10s} {'':>8s} {'':>9s} "
      f"{o['accuracy']:9.3f} {cm_pooled.sum():11,}")

print("\n  confusion matrix, row-normalised (rows = true class)\n")
cmn = cm_pooled / np.maximum(cm_pooled.sum(1, keepdims=True), 1)
print("    " + " " * 24 + "".join(f"{i:>8d}" for i in range(N_CLASSES)))
for i in range(N_CLASSES):
    print(f"    {i} {CLASS_NAMES[i]:22s}" + "".join(f"{cmn[i, j]:8.3f}"
                                                    for j in range(N_CLASSES)))

In [ ]:
# ---------------------------------------------------------------------------
#  TABLE 2 - per-class score for each fold, with mean +/- std
# ---------------------------------------------------------------------------
#  The spread matters as much as the mean. Only ~5 test users per fold have
#  Running or Bicycling, so a large std on a rare class describes the fold
#  assignment more than the classifier.
# ---------------------------------------------------------------------------

TABLE2_METRIC = "f1"          # "f1", "recall" or "precision"

per_fold = np.full((N_FOLDS, N_CLASSES), np.nan)
for i, r in enumerate(rf_results):
    per_fold[i] = class_metrics(confusion_np(r["y_true"], r["y_pred"]))[TABLE2_METRIC]

print(f"TABLE 2 - per-class {TABLE2_METRIC.upper()} across the {N_FOLDS} folds\n")
print(f"{'idx':>3}  {'activity':22s}"
      + "".join(f"{'fold ' + str(i):>9s}" for i in range(N_FOLDS))
      + f"{'mean':>9s}{'std':>8s}")
print("-" * 87)
for k in range(N_CLASSES):
    row = per_fold[:, k]
    print(f"{k:>3}  {CLASS_NAMES[k]:22s}"
          + "".join(f"{per_fold[i, k]:9.3f}" for i in range(N_FOLDS))
          + f"{np.nanmean(row):9.3f}{np.nanstd(row, ddof=1):8.3f}")
print("-" * 87)
print(f"{'':>3}  {'macro (mean of rows)':22s}"
      + "".join(f"{np.nanmean(per_fold[i]):9.3f}" for i in range(N_FOLDS))
      + f"{np.nanmean(per_fold):9.3f}{np.nanstd(np.nanmean(per_fold, 1), ddof=1):8.3f}")

print(f"\n  MEAN MACRO F1 over the 5 folds: {np.nanmean(per_fold):.4f} "
      f"+/- {np.nanstd(np.nanmean(per_fold, axis=1), ddof=1):.4f}")
worst = int(np.nanargmax(np.nanstd(per_fold, axis=0, ddof=1)))
print(f"  most fold-dependent class: {CLASS_NAMES[worst]} "
      f"({TABLE2_METRIC} {np.nanmin(per_fold[:, worst]):.3f} to "
      f"{np.nanmax(per_fold[:, worst]):.3f})")

In [ ]:
# ---------------------------------------------------------------------------
#  Feature importances - what the forest actually used
# ---------------------------------------------------------------------------
#  This is the reason to run a forest on this problem at all. The open question
#  is whether the posture classes (Lying / Sitting / Standing) carry any
#  cross-user signal. Group H - grav_dir_x/y/z and grav_mag - is the ONLY
#  feature group that speaks to posture: it is the direction of gravity in the
#  device frame. If those rank low while the posture classes still fail, the
#  ambiguity is confirmed and no amount of model capacity will fix it.
# ---------------------------------------------------------------------------
TOP_N = 25
imp = np.mean([r["importances"] for r in rf_results], axis=0)
order = np.argsort(-imp)

print(f"top {TOP_N} features (mean importance over {N_FOLDS} folds)\n")
print(f"  {'rank':>4}  {'feature':28s} {'importance':>11s}")
for rank, j in enumerate(order[:TOP_N], 1):
    bar = "#" * int(round(imp[j] / imp[order[0]] * 28))
    print(f"  {rank:>4}  {FEATURE_NAMES[j]:28s} {imp[j]:11.5f}  {bar}")

print("\nimportance by feature group\n")
groups = {
    "orientation (gravity direction)": lambda n: n.startswith("grav_"),
    "rotation-invariant magnitudes":   lambda n: n.startswith(("acc_mag", "gyro_mag")),
    "vertical / horizontal split":     lambda n: n.startswith(("acc_vert", "acc_horiz")),
    "raw acc channels":                lambda n: n.startswith(("acc_x", "acc_y", "acc_z")),
    "raw gyro channels":               lambda n: n.startswith(("gyro_x", "gyro_y", "gyro_z")),
    "cross-axis correlation":          lambda n: n.startswith("corr_"),
}
print(f"  {'group':34s} {'n':>4} {'total':>9} {'mean/feature':>14}")
claimed = np.zeros(len(FEATURE_NAMES), dtype=bool)
for name, test in groups.items():
    m = np.array([test(n) for n in FEATURE_NAMES])
    claimed |= m
    print(f"  {name:34s} {m.sum():>4} {imp[m].sum():9.4f} {imp[m].mean():14.5f}")
if (~claimed).any():                       # empty when the groups cover everything
    print(f"  {'(everything else)':34s} {(~claimed).sum():>4} "
          f"{imp[~claimed].sum():9.4f} {imp[~claimed].mean():14.5f}")

kinds = {"spectral": ("dom_freq", "dom_power", "spec_"), "periodicity": ("_ac_",),
         "band energy": ("_band",), "dynamics (jerk)": ("jerk_",)}
print()
for name, pre in kinds.items():
    m = np.array([any(p in n for p in pre) for n in FEATURE_NAMES])
    print(f"  {name:34s} {m.sum():>4} {imp[m].sum():9.4f} {imp[m].mean():14.5f}")